In [4]:
import sys
sys.path.insert(0, r"PROJECT_ROOT")

from ts_init import fetch_stock_data
from indicators import IndicatorFactory
from openai import OpenAI
import pandas as pd
import numpy as np

print("✅ 组合管理器初始化完成")

✅ 组合管理器初始化完成


In [5]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

# ⚠️ 请替换成你的真实 DeepSeek Key
DS_KEY = "your_deepseek_api_key"

client = OpenAI(
    api_key=DS_KEY,
    base_url="https://api.deepseek.com",
    default_headers={"Content-Type": "application/json; charset=utf-8"}
)

# 测试连通性（用英文，避免编码问题）
try:
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": "Reply OK"}],
        max_tokens=10
    )
    print("✅ DeepSeek 连通成功")
except Exception as e:
    print("❌ DeepSeek 连接失败:", e)

✅ DeepSeek 连通成功


In [6]:
def safe_text(text):
    """安全处理中文字符，避免编码错误"""
    if isinstance(text, str):
        return text.encode('utf-8', errors='ignore').decode('utf-8')
    return str(text)

def analyze_single_stock(ts_code, position_ratio):
    df = fetch_stock_data(ts_code)
    df = IndicatorFactory.add_all_indicators(df.copy())
    
    L = df.dropna().iloc[-1]
    trend = "Bullish" if L["ma5"] > L["ma20"] else "Bearish"
    rsi_status = "Overbought" if L["rsi14"] > 70 else ("Oversold" if L["rsi14"] < 30 else "Neutral")
    
    score = 5
    if trend == "Bullish": score += 2
    if rsi_status == "Neutral": score += 1
    if rsi_status == "Overbought": score -= 2
    
    summary = f"{ts_code} | Weight:{position_ratio*100:.0f}% | Trend:{trend} | RSI:{rsi_status} | Score:{score}/10"
    return safe_text(summary), score

def portfolio_health_check(portfolio_dict):
    print("🔄 Starting portfolio health check...\n")
    
    total_score = 0
    reports = []
    
    for code, weight in portfolio_dict.items():
        report, score = analyze_single_stock(code, weight)
        reports.append(report)
        total_score += score * weight
        print(f"✅ {report}")
    
    # 使用英文 prompt 避免编码问题
    prompt = f"""My portfolio: {portfolio_dict}
Individual analysis:
{chr(10).join(reports)}

Please provide:
1. Overall assessment of my portfolio
2. Biggest risk point
3. One actionable suggestion
Answer in Chinese."""
    
    try:
        ai_review = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.7
        ).choices[0].message.content
    except Exception as e:
        ai_review = f"AI analysis failed: {str(e)}"
    
    print("\n" + "="*50)
    print(f"📊 Portfolio Health Score: {round(total_score, 1)}/10")
    print("="*50)
    print("\n🤖 AI Portfolio Review:")
    print(safe_text(ai_review))
    
    return reports

print("✅ 工具函数定义完成")

✅ 工具函数定义完成


In [7]:
# 你的持仓字典
my_portfolio = {
    "600519.SH": 0.60,   # 茅台 60%
    "000001.SZ": 0.40    # 平安银行 40%
}

# 执行体检
portfolio_health_check(my_portfolio)

🔄 Starting portfolio health check...

✅ 从缓存加载 600519.SH，共 339 行
✅ 600519.SH | Weight:60% | Trend:Bullish | RSI:Neutral | Score:8/10
✅ 从缓存加载 000001.SZ，共 339 行
✅ 000001.SZ | Weight:40% | Trend:Bullish | RSI:Neutral | Score:8/10

📊 Portfolio Health Score: 8.0/10

🤖 AI Portfolio Review:
好的，以下是对您当前投资组合的分析与建议：

---

### 1. 整体评估

您的组合由 **贵州茅台（600519.SH，60%）** 和 **平安银行（000001.SZ，40%）** 构成，整体呈现 **“消费龙头+金融蓝筹”** 的经典配置。  
- **趋势与评分**：两只个股均为 **看涨趋势**，且评分均为8/10，表明当前技术面与基本面支撑较好，市场情绪积极。  
- **风险分散度**：权重集中在两个行业（白酒与银行），**行业相关性较低**（消费周期 vs 金融周期），具备一定对冲效果。  
- **综合结论**：组合风格偏稳健，适合中长期持有，短期波动可控，但需警惕行业集中风险。

---

### 2. 最大风险点

**行业集中度过高，且均受宏观经济与政策影响显著。**  
- **贵州茅台**：若消费疲软、白酒行业监管趋严（如消费税改革），或市场风格转向成长股，可能拖累股价。  
- **平安银行**：受利率政策、房地产风险暴露、净息差收窄等影响，银行股易出现估值修复缓慢甚至下行。  
- **联动风险**：虽然行业不同，但若出现系统性风险（如经济衰退、流动性收紧），两者可能同步下跌，失去分散作用。

---

### 3. 一个可操作建议

**动态平衡权重，适度降低单一押注，引入防御性资产。**  
- 具体操作：将 **10%-15%** 的仓位从两只个股中均分减仓，配置至 **低波动红利ETF（如中证红利指数基金）** 或 **国债ETF**。  
- 理由：  
  - 降低行业集中风险，增强组合抗跌性。  
  - 红利资产在当前震荡市中提供稳定分红，国债ETF对冲

['600519.SH | Weight:60% | Trend:Bullish | RSI:Neutral | Score:8/10',
 '000001.SZ | Weight:40% | Trend:Bullish | RSI:Neutral | Score:8/10']

In [8]:
def strategy_filter(stock_list, condition="ma5>ma20"):
    results = []
    print(f"🔍 Screening stocks with condition: {condition}")
    
    for code in stock_list:
        try:
            df = fetch_stock_data(code)
            df = IndicatorFactory.add_all_indicators(df.copy())
            L = df.dropna().iloc[-1]
            
            if condition == "ma5>ma20" and L["ma5"] > L["ma20"]:
                results.append(f"{code} (MA5:{L['ma5']:.1f} > MA20:{L['ma20']:.1f})")
            elif condition == "rsi<30" and L["rsi14"] < 30:
                results.append(f"{code} (RSI:{L['rsi14']:.1f})")
        except:
            pass
            
    print(f"\n✅ Found {len(results)} stocks matching criteria:")
    for r in results:
        print(r)
    return results

# 测试
watchlist = ["600519.SH", "000001.SZ", "300750.SZ", "600036.SH"]
strategy_filter(watchlist, condition="ma5>ma20")

🔍 Screening stocks with condition: ma5>ma20
✅ 从缓存加载 600519.SH，共 339 行
✅ 从缓存加载 000001.SZ，共 339 行
🔄 正在从 Tushare 拉取 300750.SZ...
✅ 300750.SZ 数据已保存，共 339 行
🔄 正在从 Tushare 拉取 600036.SH...
✅ 600036.SH 数据已保存，共 339 行

✅ Found 2 stocks matching criteria:
600519.SH (MA5:1457.1 > MA20:1448.5)
000001.SZ (MA5:11.5 > MA20:11.4)


['600519.SH (MA5:1457.1 > MA20:1448.5)', '000001.SZ (MA5:11.5 > MA20:11.4)']